# KhabarNazar: Roman Urdu dataset, train / validation / test split

This notebook splits the 8,267-article Roman Urdu dataset into **train (80%)**, **validation (10%)** and **test (10%)**.

Two rules:

1. **Stratified by source and label.** Each split has the same mix of Bend the Truth, FIRE, Ax-to-Grind and Hook and Bait, real and fake.
2. **Grouped by near-duplicates.** Some articles are near-copies of each other. In Bend the Truth and FIRE, most fake articles are a *one-word edit* of a real article (for example "motorway" changed to "GT"). If one copy were in train and the other in test, the test score would be inflated. Near-copies are therefore joined into a group, and a group always goes into one split.

Run it from the `notebooks/` folder with the dataset at `data/khabarnazar_roman_urdu_dataset.csv`. Output: `data/splits/{train,val,test}.csv` and `outputs/split_report.json`.

## 1. Load the dataset

In [ ]:
import re
import json
from collections import defaultdict
from pathlib import Path

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

DATA_PATH = Path("../data/khabarnazar_roman_urdu_dataset.csv")
OUT_DIR = Path("../data/splits")
RESULTS_DIR = Path("../outputs")
SEED = 42
JACCARD_THRESHOLD = 0.5   # two articles with >= 50% shared word-trigrams count as near-copies
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(df.shape)
print(df.groupby(["source_dataset", "label"]).size())

## 2. Find near-duplicate articles

Each article (Urdu script) is turned into a set of word trigrams (three-word sequences). Two articles whose trigram sets have a Jaccard similarity of 0.5 or more are treated as near-copies.

In [ ]:
def clean(text):
    return re.sub(r"[^\w]+", " ", str(text)).strip()

def trigrams(text):
    words = clean(text).split()
    if len(words) < 3:
        return {tuple(words)}
    return {tuple(words[i:i + 3]) for i in range(len(words) - 2)}

shingles = df["original_text"].map(trigrams).tolist()

# inverted index: trigram -> rows that contain it
index = defaultdict(set)
for row, grams in enumerate(shingles):
    for g in grams:
        index[g].add(row)

pairs = []
for i, grams in enumerate(shingles):
    shared = defaultdict(int)
    for g in grams:
        rows = index[g]
        if len(rows) < 50:            # skip very common phrases, they say nothing about copying
            for j in rows:
                if j > i:
                    shared[j] += 1
    for j, n in shared.items():
        jaccard = n / len(grams | shingles[j])
        if jaccard >= JACCARD_THRESHOLD:
            pairs.append((i, j, jaccard))

pairs = pd.DataFrame(pairs, columns=["i", "j", "jaccard"])
pairs["source_i"] = df["source_dataset"].values[pairs["i"]]
pairs["source_j"] = df["source_dataset"].values[pairs["j"]]
pairs["same_label"] = df["label"].values[pairs["i"]] == df["label"].values[pairs["j"]]
print(f"{len(pairs)} near-duplicate pairs")
print(pairs.groupby(["source_i", "source_j", "same_label"]).size())

## 3. Join near-copies into groups

If A is a copy of B and B is a copy of C, all three go into one group (union-find).

In [ ]:
parent = list(range(len(df)))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

for i, j in zip(pairs["i"], pairs["j"]):
    parent[find(i)] = find(j)

roots = pd.Series([find(i) for i in range(len(df))])
# name each group after its smallest article id, so the split is identical on every run
df["group_id"] = df["id"].groupby(roots.values).transform("min").values
sizes = df["group_id"].value_counts()
print("groups with more than one article:", (sizes > 1).sum())
print("largest group:", sizes.max())

## 4. Split

`StratifiedGroupKFold` makes 10 folds that keep the source/label mix and never break a group. Fold 0 is the test set, fold 1 the validation set, folds 2 to 9 the training set. The `assert` proves that no group is split across sets.

In [ ]:
# 10 folds stratified by source + label; a group always lands in one fold.
# fold 0 -> test, fold 1 -> validation, folds 2-9 -> train
strata = df["source_dataset"] + "_" + df["label"]
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
df["fold"] = -1
for fold, (_, idx) in enumerate(sgkf.split(df, strata, groups=df["group_id"])):
    df.loc[df.index[idx], "fold"] = fold

df["split"] = df["fold"].map(lambda f: "test" if f == 0 else ("val" if f == 1 else "train"))

# leakage check: no group may appear in two splits
assert df.groupby("group_id")["split"].nunique().max() == 1, "a near-duplicate group is split across sets!"
print("leakage check passed")
print(pd.crosstab([df["source_dataset"], df["label"]], df["split"], margins=True))

## 5. Save

The CSVs hold the full article text, so they stay **out of GitHub** (see `.gitignore`). `split_ids.csv` holds only ids and is safe to commit, so anyone with the original data can rebuild the exact same split.

In [ ]:
cols = ["id", "source_dataset", "original_text", "label", "roman_text", "group_id"]
for name in ["train", "val", "test"]:
    part = df[df["split"] == name][cols].sample(frac=1, random_state=SEED)
    part.to_csv(OUT_DIR / f"{name}.csv", index=False)
    print(name, len(part))

report = {
    "seed": SEED,
    "jaccard_threshold": JACCARD_THRESHOLD,
    "total_rows": int(len(df)),
    "near_duplicate_pairs": int(len(pairs)),
    "groups_with_more_than_one_article": int((sizes > 1).sum()),
    "rows": {s: int((df["split"] == s).sum()) for s in ["train", "val", "test"]},
    "by_source_label": {
        s: {f"{src}/{lab}": int(n) for (src, lab), n in
            df[df["split"] == s].groupby(["source_dataset", "label"]).size().items()}
        for s in ["train", "val", "test"]
    },
}
(RESULTS_DIR / "split_report.json").write_text(json.dumps(report, indent=2))
# the split assignment by id only (no article text), safe to commit to GitHub
df[["id", "split"]].to_csv(RESULTS_DIR / "split_ids.csv", index=False)
print(json.dumps(report["rows"], indent=2))